# ** The "Sentinel" Server Log Analyzer**

## **Introduction**

Welcome to the **Site Reliability Engineering** (SRE) team at TechCorp. Our main production server ("**The Sentinel**") has been acting up; users are reporting **slow load times**, and security has flagged a **potential brute-force attack**.

The server logs are being dumped into a **raw text stream**, but no one has analyzed them yet. We need you to build a Python program to parse these logs, identify malicious IPs, and calculate system performance metrics.

![image.png](https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExbmNlZmFhbGJxdmY2YTNpcWUycDBkaWF0dWZ6NWoxM2ppMW56eHhsMSZlcD12MV9naWZzX3NlYXJjaCZjdD1n/FkLGEwAvYbCcrlJp3s/giphy.gif)

## **Problem Statement**

You have a raw string of server logs containing IPs, request methods (GET/POST), endpoints, status codes, and response times. The data is messy and unstructured.

**Your Mission:** Build a **Log Analysis Pipeline** that:

1. **Ingests** raw data from a simulated stream.
2. **Parses & Transforms** the unstructured text into a clean List of Dictionaries.
3. **Sanitizes** the data (handling unit conversion and type casting).
4. **Audits Security:** Detects potential attacks based on error codes.
5. **Audits Performance:** Calculates average latency and identifies bottlenecks.
6. **Reports:** outputs a Markdown-formatted summary.

![Image.png](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQI5NsOcdMnWdvrkppAiVXs5yzbD-R3IyQZJw&s)

---


## **The Data (Raw Logs)**

The data stream is messy. It contains inconsistent spacing and multiple fields separated by different delimiters.

```python
raw_logs = """
192.168.1.10 - GET /login 200 45ms;
192.168.1.11 - POST /login 401 120ms;
10.0.0.5 - GET /dashboard 500 800ms;
192.168.1.11 - POST /login 401 115ms;
192.168.1.10 - GET /home 200 30ms;
10.0.0.15 - GET /api/data 200 150ms;
192.168.1.11 - POST /login 401 110ms;
10.0.0.5 - GET /dashboard 503 2000ms;
192.168.1.20 - POST /upload 201 500ms;
192.168.1.11 - POST /login 401 130ms;
10.0.0.5 - GET /dashboard 500 950ms
"""

```

## **Workflow & Instructions**

![image.png](https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExenV0NnBhbTRsZm1yOGV1aG1wNWhpbG1kcm52MWxybzhueWdyZjJucCZlcD12MV9naWZzX3NlYXJjaCZjdD1n/R6xi8dXsRhIjK/giphy.gif
)

#### **Step 1: Ingestion & Pre-processing**

* **Concept:** String Cleaning
* **Task:** The `raw_logs` string has newlines and extra whitespace.
1. Clean the string to remove leading/trailing whitespace.
2. Split the string by the semicolon `;` delimiter to get individual log entries.
3. Filter out any empty strings that might result from the split.

In [3]:
# --- RAW DATA STREAM ---
# Note: Run this cell to initialize the raw_logs variable.
raw_logs = """
192.168.1.10 - GET /login 200 45ms;
192.168.1.11 - POST /login 401 120ms;
10.0.0.5 - GET /dashboard 500 800ms;
192.168.1.11 - POST /login 401 115ms;
192.168.1.10 - GET /home 200 30ms;
10.0.0.15 - GET /api/data 200 150ms;
192.168.1.11 - POST /login 401 110ms;
10.0.0.5 - GET /dashboard 503 2000ms;
192.168.1.20 - POST /upload 201 500ms;
192.168.1.11 - POST /login 401 130ms;
10.0.0.5 - GET /dashboard 500 950ms
"""

In [4]:
#  STEP 1: INGESTION
# TODO: Strip whitespace and split by ';'
# METHOD 1 

clean_data = raw_logs.strip().split(";") # Cleaning and splitting raw dataset
print(clean_data)

# Filtering out empty string
filtered_log = []                    # this will hold the clean entries
for log in clean_data:
    log = log.strip()           # clean whitespace/newlines off this entry
    if log:                 # only keep it if it's not empty
        filtered_log.append(log)

print(filtered_log)

# METHOD 2 - List Comprehension
# filtered_log = [log.strip() for log in raw_logs.split(';') if log.strip()]
#print(filtered_log)



['192.168.1.10 - GET /login 200 45ms', '\n192.168.1.11 - POST /login 401 120ms', '\n10.0.0.5 - GET /dashboard 500 800ms', '\n192.168.1.11 - POST /login 401 115ms', '\n192.168.1.10 - GET /home 200 30ms', '\n10.0.0.15 - GET /api/data 200 150ms', '\n192.168.1.11 - POST /login 401 110ms', '\n10.0.0.5 - GET /dashboard 503 2000ms', '\n192.168.1.20 - POST /upload 201 500ms', '\n192.168.1.11 - POST /login 401 130ms', '\n10.0.0.5 - GET /dashboard 500 950ms']
['192.168.1.10 - GET /login 200 45ms', '192.168.1.11 - POST /login 401 120ms', '10.0.0.5 - GET /dashboard 500 800ms', '192.168.1.11 - POST /login 401 115ms', '192.168.1.10 - GET /home 200 30ms', '10.0.0.15 - GET /api/data 200 150ms', '192.168.1.11 - POST /login 401 110ms', '10.0.0.5 - GET /dashboard 503 2000ms', '192.168.1.20 - POST /upload 201 500ms', '192.168.1.11 - POST /login 401 130ms', '10.0.0.5 - GET /dashboard 500 950ms']


#### **Step 2: Parsing & Transformation (ETL)**

* **Concept:** List/Dict Structures & String Manipulation
* **Task:** Create a list of dictionaries named `parsed_logs`. Loop through your cleaned entries and parse them.
* *Input:* `"192.168.1.10 - GET /login 200 45ms"`
* *Logic:*
1. Split the entry by `" - "` to separate the IP from the rest.
2. Split the remaining part by space `" "` to get Method, Endpoint, Status, Latency.

**Schema (Keys):**
* `ip`: **String**
* `method`: **String**
* `endpoint`: **String**
* `status`: **Integer** (Must convert from string `"200"` -> `200`)
* `latency`: **Integer** (Must remove "ms" suffix, e.g., `"45ms"` -> `45`)


In [5]:
#  STEP 2: TRANSFORMATION (ETL) 
parsed_logs = []

# TODO: Loop through log_entries.                        
for log in filtered_log:
    # Step 1: separate the IP from everything else
    ip, rest = log.split(" - ")

    # Step 2: break the rest into its four pieces
    method, endpoint, status, latency = rest.split(" ")

    # Convert status from text "200" into the number 200
    status = int(status)

    # Strip the "ms" off latency, then turn "45" into the number 45
    latency = int(latency.replace("ms", ""))

    # Bundle everything into a dictionary
    log_dict = {
        "ip": ip,
        "method": method,
        "endpoint": endpoint,
        "status": status,
        "latency": latency
    }

    # Add this dictionary to our list
    parsed_logs.append(log_dict)

print(parsed_logs)

[{'ip': '192.168.1.10', 'method': 'GET', 'endpoint': '/login', 'status': 200, 'latency': 45}, {'ip': '192.168.1.11', 'method': 'POST', 'endpoint': '/login', 'status': 401, 'latency': 120}, {'ip': '10.0.0.5', 'method': 'GET', 'endpoint': '/dashboard', 'status': 500, 'latency': 800}, {'ip': '192.168.1.11', 'method': 'POST', 'endpoint': '/login', 'status': 401, 'latency': 115}, {'ip': '192.168.1.10', 'method': 'GET', 'endpoint': '/home', 'status': 200, 'latency': 30}, {'ip': '10.0.0.15', 'method': 'GET', 'endpoint': '/api/data', 'status': 200, 'latency': 150}, {'ip': '192.168.1.11', 'method': 'POST', 'endpoint': '/login', 'status': 401, 'latency': 110}, {'ip': '10.0.0.5', 'method': 'GET', 'endpoint': '/dashboard', 'status': 503, 'latency': 2000}, {'ip': '192.168.1.20', 'method': 'POST', 'endpoint': '/upload', 'status': 201, 'latency': 500}, {'ip': '192.168.1.11', 'method': 'POST', 'endpoint': '/login', 'status': 401, 'latency': 130}, {'ip': '10.0.0.5', 'method': 'GET', 'endpoint': '/dashb

#### **Step 3: Security Audit (Logic)**

* **Concept:** Filtering & Aggregation
* **Task:** Identify potential attackers.
1. Define a function `find_failed_logins(data)`.
2. It should identify any IP address that has generated a **401 (Unauthorized)** error.
3. It should return a **Dictionary** where the Key is the IP and the Value is the **Count** of failed attempts.

In [6]:
#  STEP 3: SECURITY AUDIT 
def find_failed_logins(parsed_logs):
    # TODO: Count 401 errors per IP

    pass

# ---------- Step 3: Count failed logins (401 errors) ----------
def find_failed_logins(parsed_logs):
    failed_counts = {}
    for log in parsed_logs:
        if log["status"] == 401:
            ip = log["ip"]
            if ip in failed_counts:
                failed_counts[ip] = failed_counts[ip] + 1
            else:
                failed_counts[ip] = 1
    return failed_counts

result = find_failed_logins(parsed_logs)
print(result)
print(type(parsed_logs[0]["status"]))

{'192.168.1.11': 4}
<class 'int'>


In [50]:
find_failed_logins(parsed_logs)

{}

#### **Step 4: Performance Audit (A little bit of Math)**

* **Concept:** Aggregation & Averages
* **Task:** Analyze system health.
1. Define a function `calculate_average_latency(data)`.
2. Calculate the average latency of **all** requests.
3. **Advanced Challenge (Optional):** Calculate the average latency for **only** the `/dashboard` endpoint.

In [19]:
#  STEP 4: PERFORMANCE AUDIT 
# TODO: Calculate average latency.
    # If 'endpoint' is provided, only calculate for that specific endpoint.
def calculate_average_latency(data, endpoint=None):
    total_latency = 0          # running sum, starts at zero
    count = 0                  # how many requests we've seen
    for log in data:
        if endpoint is not None and log["endpoint"] != endpoint: # Only include matching endpoints if one is specified
            continue
        total_latency = total_latency + log["latency"]   # add this one's latency
        count = count + 1                                 # tally one more request
        if count == 0:
            return 0
        average = total_latency / count    # average latency of all request = total latency divided by count of request
        return average

average_latency = calculate_average_latency(parsed_logs)
print(average_latency)

average_latency_dashboard = calculate_average_latency(parsed_logs, "/dashboard")
print(average_latency_dashboard)


45.0
800.0


**Step 5: Reporting (Load)**

* **Concept:** Formatted Output
* **Task:** Generate a human-readable report using `f-strings`.
* Show Total Requests.
* Show Average Latency.
* List any IP with **more than 2** failed login attempts as "BANNED".


In [27]:
#  STEP 5: FINAL REPORT 

# TODO: Call functions and print formatted results
# Flag IPs with > 2 failed attempts as "BANNED"
if __name__ == "__main__":
    print("=== 🛡️ SENTINEL SYSTEM REPORT ===")
    total_request = len (parsed_logs)
    print(f"Total Request Made: {total_request}")

    average_latency = calculate_average_latency(parsed_logs)
    print(f"Average Latency per Request: { average_latency}")

    failed_logins = find_failed_logins (parsed_logs)
    for ip, count in failed_logins.items():
        if count > 2:
            print(f"{ip} is BANNED!")    


=== 🛡️ SENTINEL SYSTEM REPORT ===
Total Request Made: 11
Average Latency per Request: 45.0
192.168.1.11 is BANNED!



![image.png](https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExMjdzeHZzaGx4dHNnbHgxcmJtc3gydW0wb29mdDZrMm52bDFwcTlvNyZlcD12MV9naWZzX3NlYXJjaCZjdD1n/kVaj8JXJcDsqs/giphy.gif)